# 07 — Encoder pretraining diagnostic

Inspect the artefacts produced by `scripts/pretrain_encoder.py` for one pretraining run:

1. **Manifest** — environment + hyperparameters used.
2. **Training curve** — loss + held-out MLM accuracy over training steps.
3. **Per-token learnability** — accuracy bucketed by token family.  Tells you whether the encoder learned anything beyond positional schema memorisation.
4. **Hardest tokens** — the bottom-N predictions; these are the cases where context isn't enough and the encoder *must* be using the continuous-value channels.
5. **Encoder embedding probe** — run a few real observations through the encoder and inspect the latent dimensionality / spread.
6. **Continuous-channel sensitivity** — perturb a single continuous value (e.g. fatigue at one position) and measure how much the CLS embedding moves.  Confirms the encoder is *actually* reading the continuous channels, not just the categorical schema.

**Edit only the user-config cell below.**

## 1. User configuration

In [ ]:
from pathlib import Path

# Path to the directory produced by ``scripts/pretrain_encoder.py``.
ENCODER_DIR = Path("reports/encoder_hybrid_v1")

# How many "hardest" and "easiest" tokens to surface.
TOP_K_HARDEST = 20
TOP_K_EASIEST = 10

# Number of synthetic observations to run through the encoder for the
# sensitivity probe.  Each obs is encoded once unmodified and once with
# a perturbed continuous value to measure the CLS displacement.
N_PROBE_SAMPLES = 64

## 2. Setup

In [ ]:
import json
import os
import sys
import warnings
from contextlib import contextmanager

# Anchor to the repo root so relative paths in ENCODER_DIR work
# regardless of where Jupyter was launched.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "run_configs").is_dir():
    _root = _root.parent
if (_root / "run_configs").is_dir():
    os.chdir(_root)
    print(f"Working directory: {_root}")
else:
    print(f"WARNING: could not locate repo root from {Path.cwd()}")

warnings.filterwarnings("ignore")
os.environ["KATA_CONF_PATH"] = "/dev/null/__no_file__"

@contextmanager
def quiet():
    old = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = old

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

assert ENCODER_DIR.is_dir(), f"ENCODER_DIR does not exist: {ENCODER_DIR}"
print("Files in encoder dir:")
for p in sorted(ENCODER_DIR.iterdir()):
    print(f"  {p.name:30s} {p.stat().st_size / 1024:>8.1f} KB")

## 3. Manifest summary

Reproducibility metadata written by the pretraining script.

In [ ]:
manifest = json.loads((ENCODER_DIR / "manifest.json").read_text())
args = manifest.get("args", {})

print("=== ENV ===")
print(f"  env_config        : {manifest['env_config']}")
print(f"  hybrid mode       : {manifest.get('hybrid', 'unknown')}")
print(f"  n_technicians     : {manifest['n_techs']}")
print(f"  machine_types     : {manifest['machine_types']}")
print(f"  component_types   : {manifest['component_types']}")
print(f"  vocab_size        : {manifest['vocab_size']}")
print()
print("=== MODEL ===")
print(f"  total params      : {manifest['n_params']:,}")
print(f"  d_model           : {args.get('d_model')}")
print(f"  n_layers          : {args.get('n_layers')}")
print(f"  n_heads           : {args.get('n_heads')}")
print(f"  dropout           : {args.get('dropout')}")
print()
print("=== TRAINING ===")
print(f"  collector         : {args.get('collector')}")
print(f"  n_rollouts        : {args.get('n_rollouts'):,}")
print(f"  train_size        : {manifest['train_size']:,}")
print(f"  eval_size         : {manifest['eval_size']:,}")
print(f"  epochs            : {args.get('epochs')}")
print(f"  batch_size        : {args.get('batch_size')}")
print(f"  lr                : {args.get('lr')}")
print(f"  mask_prob         : {args.get('mask_prob')}")
print(f"  warmup_steps      : {args.get('warmup_steps')}")
print(f"  device            : {args.get('device')}")
print(f"  training duration : {manifest['training_duration_seconds']:.1f} s")
print()
print("=== RESULTS ===")
print(f"  final train loss  : {manifest['final_train_loss']:.4f}")
print(f"  final train acc   : {manifest['final_train_accuracy']:.4f}")
print(f"  held-out acc      : {manifest['held_out_mlm_accuracy']:.4f}")

## 4. Training curve

Inline render of `curve.png`.  The red dashed line is the held-out accuracy.

In [ ]:
curve_path = ENCODER_DIR / "curve.png"
if curve_path.exists():
    img = Image.open(curve_path)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    ax.axis("off")
    plt.show()
else:
    print(f"No curve at {curve_path}")

## 5. Per-token learnability

A high *overall* MLM accuracy can hide a simple failure mode: the model learns the **positional schema** (which key tokens appear at which slots) without learning anything about the content of the observation.

The fix is to look at accuracy **bucketed by token family**:

- **Schema tokens** (`SIM_TIME`, `FATIGUE`, `MACHINE_BROKEN`, `TECH_3`, …) — fixed positions, near-100% accuracy expected.  This is *not* evidence the encoder learned anything useful.
- **Content tokens** (machine types, component types, booleans like `MACHINE_BROKEN`→`TRUE/FALSE`) — these depend on the actual factory state; accuracy here is the honest signal.
- **Bucket value tokens** (`T_*`, `R_*`, `C_*`) — should have 0 evaluations in hybrid mode (the env emits `<NUM>` instead).  If they have nonzero counts, something is leaking the legacy emission path.

In [ ]:
df = pd.read_csv(ENCODER_DIR / "per_token_report.csv")
df.head()

In [ ]:
def classify(token: str) -> str:
    """Bucket a vocab token into a family for aggregate stats."""
    if token in {"<PAD>", "<UNK>", "<BOS>", "<EOS>"}:
        return "special"
    if token == "<NUM>":
        return "num_placeholder"
    if token.startswith("T_"):
        return "time_bucket"
    if token.startswith("R_"):
        return "ratio_bucket"
    if token.startswith("C_"):
        return "count_bucket"
    if token in {"TRUE", "FALSE"}:
        return "boolean"
    if token == "NONE":
        return "sentinel"
    if token.startswith("TECH_"):
        return "tech_prefix"
    if token.startswith("BROKEN_"):
        return "broken_by_type"
    if token.startswith("QC_"):
        return "queue_composition"
    if token.startswith("NEXT") and "_" in token:
        return "next_ticket_key"
    if token in {"ticket_only", "broken_machine", "factory_level", "tech_aware"}:
        return "obs_mode_value"
    if token.isupper() and "_" in token:
        return "schema_key"
    return "categorical_value"

df["family"] = df["token"].map(classify)

summary = (
    df.groupby("family")
      .agg(
          n_tokens=("token", "count"),
          n_evaluated=("n_evaluated", "sum"),
          mean_acc=("accuracy", "mean"),
          min_acc=("accuracy", "min"),
          max_acc=("accuracy", "max"),
      )
      .sort_values("mean_acc")
)
summary

In [ ]:
# Bar chart of mean accuracy by family, with n_evaluated as marker size.
fig, ax = plt.subplots(figsize=(10, 4.5))
fam_order = summary.index.tolist()
means = [summary.loc[f, "mean_acc"] for f in fam_order]
counts = [summary.loc[f, "n_evaluated"] for f in fam_order]
ax.bar(fam_order, means, color="tab:blue", edgecolor="black", linewidth=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean MLM accuracy")
ax.set_title("Per-token-family learnability")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, axis="y", alpha=0.3)
for x, m, n in zip(fam_order, means, counts):
    ax.text(x, m + 0.02, f"n={n}", ha="center", fontsize=8, color="gray")
fig.tight_layout()
plt.show()

## 6. Hardest tokens — where context is not enough

Tokens with the lowest accuracy are the cases where the surrounding schema doesn't deterministically pin the answer.  In hybrid mode, these are exactly the tokens where the encoder *must* have leaned on the parallel continuous channels (`cont_values` × `cont_kinds`) to make the prediction — they are the strongest evidence that the continuous-feature encoders did something useful.

Conversely, tokens at 100 % accuracy with many evaluations are pure positional memorisation.

In [ ]:
evaluated = df[df["n_evaluated"] > 0].copy()
print(f"=== {TOP_K_HARDEST} HARDEST tokens (lowest accuracy, ties broken by n_evaluated):")
hardest = evaluated.sort_values(["accuracy", "n_evaluated"], ascending=[True, False]).head(
    TOP_K_HARDEST
)
print(hardest[["token", "family", "n_evaluated", "accuracy"]].to_string(index=False))

print(f"\n=== {TOP_K_EASIEST} EASIEST tokens (highest accuracy, most evaluated):")
easiest = evaluated.sort_values(
    ["accuracy", "n_evaluated"], ascending=[False, False]
).head(TOP_K_EASIEST)
print(easiest[["token", "family", "n_evaluated", "accuracy"]].to_string(index=False))

## 7. Encoder embedding probe

Load the trained encoder and inspect what its CLS embedding looks like on a small batch of real observations from the same env it was trained on.

What we report:
- **Embedding dimensionality** — should match the `d_model` from the manifest.
- **Per-dimension activation statistics** (mean / std) — a collapsed encoder will show very low variance across the batch; a well-spread one will use most dimensions actively.
- **Pairwise cosine similarities** — tells you whether different observations produce distinguishable embeddings.  A median similarity near 1.0 means the encoder maps everything to the same vector (collapse); near 0 means the embeddings are well-spread.

In [ ]:
from agents.representation.mtm import MTMTrainer
from kata.core.config import KATAConfig
from kata.core.tokenizer import StateTokenizer
from kata.env import KataEnv
from kata.EntityFactories import RandomScenarioSampler
from kata.scenario import ScenarioBuilder
from agents import RandomAgent

device = torch.device("cpu")
encoder = MTMTrainer.load_encoder(ENCODER_DIR / "encoder.pt", device=device)
encoder.eval()
print(f"Encoder class: {type(encoder).__name__}")
print(f"Output dim   : {encoder.output_dim}")
print(f"Param count  : {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# Rebuild the env (must match the one used at pretraining time) and
# gather N_PROBE_SAMPLES real observations using a random agent.
env_path = Path(manifest["env_config"])
cfg = KATAConfig(**json.loads(env_path.read_text()))

rcfg = cfg.randomized_scenario
if rcfg.enabled:
    sampler = RandomScenarioSampler(cfg, rcfg, seed=0)
    factory = lambda: ScenarioBuilder(sampler.sample_config()).build()
    n_techs = rcfg.n_technicians
    machine_types = sampler.all_machine_types()
    component_types = sampler.all_component_types()
else:
    factory = lambda: ScenarioBuilder(cfg).build()
    n_techs = len(cfg.technicians)
    machine_types = sorted({m.machine_type for m in cfg.machines.values()})
    component_types = sorted(
        {c.component_type for m in cfg.machines.values() for c in m.components.values()}
    )

tokenizer = StateTokenizer.build_vocab(
    machine_types=machine_types,
    n_technicians=n_techs,
    seq_length=cfg.gym.tokenizer_seq_length,
    component_types=component_types,
    next_ticket_lookahead=cfg.gym.next_ticket_lookahead,
)
with quiet():
    env = KataEnv(scenario_factory=factory, config=cfg.gym, tokenizer=tokenizer)
    agent = RandomAgent(n_actions=env.action_space.n)
    obs, _ = env.reset(seed=0)
    obs_batch = {
        "token_ids": [], "cont_values": [], "cont_kinds": [],
    }
    while len(obs_batch["token_ids"]) < N_PROBE_SAMPLES:
        for k in obs_batch:
            obs_batch[k].append(obs[k].copy() if hasattr(obs[k], "copy") else np.array(obs[k]))
        action = agent.select_action(obs)
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            obs, _ = env.reset()

batch = {k: torch.from_numpy(np.stack(v)).to(device) for k, v in obs_batch.items()}
with torch.no_grad():
    cls, _ = encoder.encode(
        batch["token_ids"], batch["cont_values"].float(),
        batch["cont_kinds"]
    )
cls_np = cls.cpu().numpy()
print(f"CLS batch shape: {cls_np.shape}")
print(f"Per-dim mean (first 8) : {cls_np.mean(axis=0)[:8].round(3)}")
print(f"Per-dim std  (first 8) : {cls_np.std(axis=0)[:8].round(3)}")
print(f"Across-dim std overall : {cls_np.std(axis=0).mean():.4f}")

In [ ]:
# Pairwise cosine similarities.  A median near 0 = embeddings are
# well-spread; median near 1 = collapse.
normed = cls_np / (np.linalg.norm(cls_np, axis=1, keepdims=True) + 1e-12)
sims = normed @ normed.T
iu = np.triu_indices_from(sims, k=1)
off_diag = sims[iu]

print(
    f"Pairwise cosine similarity over {len(off_diag)} pairs: "
    f"mean={off_diag.mean():.3f}  median={np.median(off_diag):.3f}  "
    f"min={off_diag.min():.3f}  max={off_diag.max():.3f}"
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(off_diag, bins=40, color="tab:blue", edgecolor="black", alpha=0.7)
ax.axvline(off_diag.mean(), color="red", linewidth=1.5, label=f"mean={off_diag.mean():.3f}")
ax.set_xlabel("pairwise cosine similarity")
ax.set_ylabel("count")
ax.set_title("Distribution of CLS embedding similarities (off-diagonal)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Continuous-channel sensitivity probe

The acid test for whether the hybrid encoder is *actually* using its PLE / Time2Vec / Fourier branches: take a real observation, encode it, then **perturb the raw value at one continuous position** (e.g. change a fatigue level from 0.1 to 0.9) and re-encode.

If the encoder is reading the continuous channels, the two CLS vectors will differ meaningfully.  If the encoder learned only the categorical schema (and ignored the continuous side), the two vectors will be nearly identical.

We report the *mean cosine distance* induced by such a perturbation, averaged over the batch.  Useful interpretation:

- **< 0.01** — the encoder essentially ignores the perturbed channel.
- **0.01 – 0.05** — the channel contributes weakly.
- **> 0.05** — substantial sensitivity; the channel is being used.

In [ ]:
from agents.networks.continuous_features import ContKind

def probe_sensitivity(batch, kind_to_perturb: int, new_value: float):
    """Encode the batch unmodified and with all kind=K positions set to ``new_value``.

    Returns the mean cosine distance between the two CLS vectors over
    the batch, and the fraction of samples that actually had at least
    one position of that kind (otherwise the perturbation is a no-op).
    """
    with torch.no_grad():
        cls_orig, _ = encoder.encode(
            batch["token_ids"], batch["cont_values"].float(), batch["cont_kinds"]
        )
        perturbed_values = batch["cont_values"].clone().float()
        mask = (batch["cont_kinds"] == kind_to_perturb)
        perturbed_values[mask] = float(new_value)
        cls_perturbed, _ = encoder.encode(
            batch["token_ids"], perturbed_values, batch["cont_kinds"]
        )
    sims = torch.nn.functional.cosine_similarity(cls_orig, cls_perturbed, dim=-1)
    dists = (1.0 - sims).cpu().numpy()
    affected = mask.any(dim=-1).float().mean().item()
    return dists.mean(), dists.std(), affected

probes = [
    ("ratio_PLE       (e.g. fatigue, knowledge, match)", ContKind.RATIO_PLE, 0.9),
    ("count_PLE       (buffers, queue size, total processed)", ContKind.COUNT_PLE, 50.0),
    ("Time2Vec        (recent ticket age, ETA, LAST_AGE)", ContKind.TIME2VEC, 5000.0),
    ("Fourier         (SIM_TIME, long-horizon hazards)", ContKind.FOURIER, 100_000.0),
]
print(f"{'channel':<55}{'mean dist':>12}{'std':>10}{'pct affected':>15}")
print("-" * 92)
for label, kind, val in probes:
    mean_d, std_d, frac = probe_sensitivity(batch, kind, val)
    print(f"  {label:<53}{mean_d:>12.4f}{std_d:>10.4f}{frac * 100:>13.1f}%")

**Reading the table:**  for each continuous channel, every position routed to it is overwritten with ``new_value`` and the resulting CLS embedding is compared to the unmodified one via cosine distance.  ``pct affected`` reports how many samples in the batch actually have a position of that kind — if it's 100 % but the mean distance is essentially 0, the encoder isn't using that channel.

Compare these numbers across channels: if (say) `ratio_PLE` produces a 0.10 mean distance while `Fourier` produces 0.001, the encoder is reading the ratio channel but ignoring the long-horizon time signal.  That's a diagnostic that points at a tuning issue (Fourier ``sigma`` / ``input_scale``) or at a downstream-loss issue (the MLM objective never asked the encoder to depend on long-horizon time).